In [1]:
import os, sys, torch, numpy as np, zarr, time
from PIL import Image

RDT2_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
DATASET    = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
NORM_PATH  = "/home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt"
CHECKPOINT = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-5000"
PORT       = '/dev/ttyACM1'
SPEED      = 15

# ✅ FIX 1: CORRECT instruction (lowercase, no period)
INSTRUCTION = "pick up the marker and place it in the box"

sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))
sys.path.insert(0, os.path.join(RDT2_DIR, 'models'))
os.chdir(RDT2_DIR)

os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']          = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']           = 'false'

print("=" * 60)
print("✅ FIX 1: Instruction corrected")
print(f"   '{INSTRUCTION}'")
print("✅ FIX 2: Using umi_normalizer_official.pt")
print("✅ FIX 3: Paths configured")
print("=" * 60)

✅ FIX 1: Instruction corrected
   'pick up the marker and place it in the box'
✅ FIX 2: Using umi_normalizer_official.pt
✅ FIX 3: Paths configured


In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from models.multivqvae import MultiVQVAE
from models.normalizer.normalizer import LinearNormalizer

torch.cuda.empty_cache()

print("[1/5] Loading base model...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map="cuda"
)
print("      ✅ Base model loaded")

print("[2/5] Loading fine-tuned adapter...")
model = PeftModel.from_pretrained(base, CHECKPOINT)
model.eval()
print("      ✅ Adapter loaded")

print("[3/5] Loading processor...")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True
    
)
print("      ✅ Processor loaded")

print("[4/5] Loading VAE...")
vae = MultiVQVAE.from_pretrained(
    "robotics-diffusion-transformer/RVQActionTokenizer"
).eval().to(device="cuda", dtype=torch.float32)
print("      ✅ VAE loaded")

print("[5/5] Loading normalizer...")
# ✅ FIX 2: Use OFFICIAL normalizer (not normalizer.pt)
normalizer = LinearNormalizer.load(NORM_PATH)
print(f"      ✅ Normalizer loaded: {type(normalizer).__name__}")

# Verify normalizer has correct keys
for key in ['action', 'robot0_eef_pos']:
    try:
        n = normalizer[key]
        print(f"      Key '{key}': ✅ {type(n).__name__}")
    except:
        print(f"      Key '{key}': ❌ Not found")

print("\n✅ ALL MODELS LOADED!")

/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/5] Loading base model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.46it/s]


      ✅ Base model loaded
[2/5] Loading fine-tuned adapter...
      ✅ Adapter loaded
[3/5] Loading processor...
      ✅ Processor loaded
[4/5] Loading VAE...
      ✅ VAE loaded
[5/5] Loading normalizer...
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
      ✅ Normalizer loaded: LinearNormalizer
      Key 'action': ✅ SingleFieldLinearNormalizer
      Key 'robot0_eef_pos': ✅ SingleFieldLinearNormalizer

✅ ALL MODELS LOADED!


In [3]:
root = zarr.open(DATASET, mode='r')
total_frames = len(root['data']['camera0_rgb'])

valid_action_id_length = (
    vae.pos_id_len + vae.rot_id_len + vae.grip_id_len
)

print(f"✅ Dataset loaded")
print(f"   Total frames:      {total_frames:,}")
print(f"   VAE token length:  {valid_action_id_length}")
print(f"   Image shape:       {root['data']['camera0_rgb'][0].shape}")

✅ Dataset loaded
   Total frames:      78,018
   VAE token length:  27
   Image shape:       (224, 224, 3)


In [ ]:
from utils import batch_predict_action

def rot6d_to_euler(rot6d):
    """Convert 6D rotation to Euler angles (degrees)"""
    a1 = rot6d[:3]
    a2 = rot6d[3:]
    b1 = a1 / (np.linalg.norm(a1) + 1e-8)
    b2 = a2 - np.dot(b1, a2) * b1
    b2 = b2 / (np.linalg.norm(b2) + 1e-8)
    b3 = np.cross(b1, b2)
    R  = np.stack([b1, b2, b3], axis=-1)
    sy = np.sqrt(R[0,0]**2 + R[1,0]**2)
    if sy > 1e-6:
        rx = np.degrees(np.arctan2( R[2,1], R[2,2]))
        ry = np.degrees(np.arctan2(-R[2,0], sy))
        rz = np.degrees(np.arctan2( R[1,0], R[0,0]))
    else:
        rx = np.degrees(np.arctan2(-R[1,2], R[1,1]))
        ry = np.degrees(np.arctan2(-R[2,0], sy))
        rz = 0.0
    return np.array([rx, ry, rz])


def get_action(frame_np):
    """
    Convert frame to M750 robot action
    Input:  frame_np - (H, W, 3) uint8 numpy array
    Output: coords [x,y,z,rx,ry,rz], grip_pct [0-100]
    """
    # Prepare image tensor
    img   = np.array(
        Image.fromarray(frame_np).resize((384, 384)),
        dtype=np.uint8
    )
    img_t = torch.from_numpy(img).unsqueeze(0)  # (1, 384, 384, 3)
    
    # Run inference with FIXED instruction
    result = batch_predict_action(
        model, processor, vae, normalizer,
        examples=[{
            "obs":  {"camera0_rgb": img_t},
            "meta": {"num_camera": 1}
        }],
        valid_action_id_length=valid_action_id_length,
        apply_jpeg_compression=True,
        instruction=INSTRUCTION  # ✅ FIX 1: correct instruction
    )
    
    # Move to CPU
    ac = result["action_pred"][0].cpu().detach()  # (24, 20)
    
    # Rescale gripper
    ac[:, 9] = ac[:, 9] / 0.088 * 0.1
    
    # Extract first timestep
    pos  = ac[0, 0:3].float().numpy()       # x, y, z (meters)
    r6d  = ac[0, 3:9].float().numpy()       # 6D rotation
    grip = float(ac[0, 9].float().item())   # gripper (meters)
    
    # Convert rotation 6D → Euler angles
    eul = rot6d_to_euler(r6d)
    
    # Convert gripper → percentage
    g_pct = int(np.clip(grip / 0.1 * 100, 0, 100))
    
    # Convert position meters → millimeters for M750
    coords = [
        round(float(pos[0]) * 1000, 1),  # x mm
        round(float(pos[1]) * 1000, 1),  # y mm
        round(float(pos[2]) * 1000, 1),  # z mm
        round(float(eul[0]), 1),          # rx deg
        round(float(eul[1]), 1),          # ry deg
        round(float(eul[2]), 1),          # rz deg
    ]
    
    return coords, g_pct



def is_safe(coords):
    """M750 workspace safety check"""
    return (
        -400 <= coords[0] <= 400  and
        -400 <= coords[1] <= 400  and
          50 <= coords[2] <= 600  and
        -180 <= coords[3] <= 180  and
        -180 <= coords[4] <= 180  and
        -180 <= coords[5] <= 180
    )

print("✅ Helper functions defined")
print(f"   get_action()   - converts frame → M750 command")
print(f"   is_safe()      - checks workspace bounds")
print(f"   rot6d_to_euler() - converts 6D rotation")

✅ Helper functions defined
   get_action()   - converts frame → M750 command
   is_safe()      - checks workspace bounds
   rot6d_to_euler() - converts 6D rotation


In [6]:
print("=" * 60)
print("🔧 FIX: Using DATASET stats for unnormalization")
print("=" * 60)

# Compute stats directly from YOUR dataset
root    = zarr.open(DATASET, mode='r')
pos_all = np.array(root['data']['robot0_eef_pos'])
rot_all = np.array(root['data']['robot0_eef_rot_axis_angle'])
grp_all = np.array(root['data']['robot0_gripper_width']).reshape(-1, 1)

actions_all = np.concatenate([pos_all, rot_all, grp_all], axis=-1)

ACTION_MEAN = actions_all.mean(axis=0)
ACTION_STD  = actions_all.std(axis=0)
ACTION_MIN  = actions_all.min(axis=0)
ACTION_MAX  = actions_all.max(axis=0)

# Prevent divide by zero
ACTION_STD = np.where(ACTION_STD < 1e-8, 1.0, ACTION_STD)

print("Dataset action statistics:")
print(f"  Mean: {np.round(ACTION_MEAN, 4)}")
print(f"  Std:  {np.round(ACTION_STD,  4)}")
print(f"  Min:  {np.round(ACTION_MIN,  4)}")
print(f"  Max:  {np.round(ACTION_MAX,  4)}")
print(f"\nPosition ranges in METERS:")
print(f"  X: [{ACTION_MIN[0]:.3f}, {ACTION_MAX[0]:.3f}]m "
      f"→ [{ACTION_MIN[0]*1000:.0f}, {ACTION_MAX[0]*1000:.0f}]mm")
print(f"  Y: [{ACTION_MIN[1]:.3f}, {ACTION_MAX[1]:.3f}]m "
      f"→ [{ACTION_MIN[1]*1000:.0f}, {ACTION_MAX[1]*1000:.0f}]mm")
print(f"  Z: [{ACTION_MIN[2]:.3f}, {ACTION_MAX[2]:.3f}]m "
      f"→ [{ACTION_MIN[2]*1000:.0f}, {ACTION_MAX[2]*1000:.0f}]mm")
print(f"  Gripper: [{ACTION_MIN[6]*1000:.1f}, {ACTION_MAX[6]*1000:.1f}]mm")

total_frames = len(root['data']['camera0_rgb'])
print(f"\nTotal frames: {total_frames:,}")
print("✅ Dataset stats computed!")

🔧 FIX: Using DATASET stats for unnormalization
Dataset action statistics:
  Mean: [ 0.0077 -0.1062  0.1275 -2.1636 -0.1622  0.0445  0.0265]
  Std:  [0.157  0.1385 0.104  1.3838 0.8236 0.1868 0.0119]
  Min:  [-0.7661 -0.874  -0.0393 -3.1406 -3.1398 -0.8489  0.0065]
  Max:  [0.3858 0.291  0.3783 3.1409 3.1406 1.1756 0.0438]

Position ranges in METERS:
  X: [-0.766, 0.386]m → [-766, 386]mm
  Y: [-0.874, 0.291]m → [-874, 291]mm
  Z: [-0.039, 0.378]m → [-39, 378]mm
  Gripper: [6.5, 43.8]mm

Total frames: 78,018
✅ Dataset stats computed!


In [7]:
print("=" * 60)
print("🔧 FIX: Using DATASET stats for unnormalization")
print("=" * 60)

# Compute stats directly from YOUR dataset
root    = zarr.open(DATASET, mode='r')
pos_all = np.array(root['data']['robot0_eef_pos'])
rot_all = np.array(root['data']['robot0_eef_rot_axis_angle'])
grp_all = np.array(root['data']['robot0_gripper_width']).reshape(-1, 1)

actions_all = np.concatenate([pos_all, rot_all, grp_all], axis=-1)

ACTION_MEAN = actions_all.mean(axis=0)
ACTION_STD  = actions_all.std(axis=0)
ACTION_MIN  = actions_all.min(axis=0)
ACTION_MAX  = actions_all.max(axis=0)

# Prevent divide by zero
ACTION_STD = np.where(ACTION_STD < 1e-8, 1.0, ACTION_STD)

print("Dataset action statistics:")
print(f"  Mean: {np.round(ACTION_MEAN, 4)}")
print(f"  Std:  {np.round(ACTION_STD,  4)}")
print(f"  Min:  {np.round(ACTION_MIN,  4)}")
print(f"  Max:  {np.round(ACTION_MAX,  4)}")
print(f"\nPosition ranges in METERS:")
print(f"  X: [{ACTION_MIN[0]:.3f}, {ACTION_MAX[0]:.3f}]m "
      f"→ [{ACTION_MIN[0]*1000:.0f}, {ACTION_MAX[0]*1000:.0f}]mm")
print(f"  Y: [{ACTION_MIN[1]:.3f}, {ACTION_MAX[1]:.3f}]m "
      f"→ [{ACTION_MIN[1]*1000:.0f}, {ACTION_MAX[1]*1000:.0f}]mm")
print(f"  Z: [{ACTION_MIN[2]:.3f}, {ACTION_MAX[2]:.3f}]m "
      f"→ [{ACTION_MIN[2]*1000:.0f}, {ACTION_MAX[2]*1000:.0f}]mm")
print(f"  Gripper: [{ACTION_MIN[6]*1000:.1f}, {ACTION_MAX[6]*1000:.1f}]mm")

total_frames = len(root['data']['camera0_rgb'])
print(f"\nTotal frames: {total_frames:,}")
print("✅ Dataset stats computed!")

🔧 FIX: Using DATASET stats for unnormalization
Dataset action statistics:
  Mean: [ 0.0077 -0.1062  0.1275 -2.1636 -0.1622  0.0445  0.0265]
  Std:  [0.157  0.1385 0.104  1.3838 0.8236 0.1868 0.0119]
  Min:  [-0.7661 -0.874  -0.0393 -3.1406 -3.1398 -0.8489  0.0065]
  Max:  [0.3858 0.291  0.3783 3.1409 3.1406 1.1756 0.0438]

Position ranges in METERS:
  X: [-0.766, 0.386]m → [-766, 386]mm
  Y: [-0.874, 0.291]m → [-874, 291]mm
  Z: [-0.039, 0.378]m → [-39, 378]mm
  Gripper: [6.5, 43.8]mm

Total frames: 78,018
✅ Dataset stats computed!


In [8]:
from utils import batch_predict_action

def rot6d_to_euler(rot6d):
    """Convert 6D rotation to Euler angles (degrees)"""
    a1 = rot6d[:3]
    a2 = rot6d[3:]
    b1 = a1 / (np.linalg.norm(a1) + 1e-8)
    b2 = a2 - np.dot(b1, a2) * b1
    b2 = b2 / (np.linalg.norm(b2) + 1e-8)
    b3 = np.cross(b1, b2)
    R  = np.stack([b1, b2, b3], axis=-1)
    sy = np.sqrt(R[0,0]**2 + R[1,0]**2)
    if sy > 1e-6:
        rx = np.degrees(np.arctan2( R[2,1], R[2,2]))
        ry = np.degrees(np.arctan2(-R[2,0], sy))
        rz = np.degrees(np.arctan2( R[1,0], R[0,0]))
    else:
        rx = np.degrees(np.arctan2(-R[1,2], R[1,1]))
        ry = np.degrees(np.arctan2(-R[2,0], sy))
        rz = 0.0
    return np.array([rx, ry, rz])


def get_action_v2(frame_np):
    """
    FIXED: Uses dataset statistics for unnormalization
    instead of official normalizer
    """
    # Prepare image
    img   = np.array(
        Image.fromarray(frame_np).resize((384, 384)),
        dtype=np.uint8
    )
    img_t = torch.from_numpy(img).unsqueeze(0)

    # Run inference - still use official normalizer for model
    # (model expects it), but we manually unnormalize after
    result = batch_predict_action(
        model, processor, vae, normalizer,
        examples=[{
            "obs":  {"camera0_rgb": img_t},
            "meta": {"num_camera": 1}
        }],
        valid_action_id_length=valid_action_id_length,
        apply_jpeg_compression=True,
        instruction=INSTRUCTION
    )

    # Get raw normalized output
    ac = result["action_pred"][0].cpu().detach()  # (24, 20)

    # Rescale gripper
    ac[:, 9] = ac[:, 9] / 0.088 * 0.1

    # Get first timestep raw values
    raw = ac[0, :7].float().numpy()  # [x, y, z, r1, r2, r3, r4, r5, r6, grip]
    r6d = ac[0, 3:9].float().numpy()

    print(f"  Raw model output (normalized): {np.round(raw, 4)}")

    # ✅ MANUALLY UNNORMALIZE using DATASET stats
    unnorm = raw * ACTION_STD[:7] + ACTION_MEAN[:7]
    unnorm = np.clip(unnorm, ACTION_MIN[:7], ACTION_MAX[:7])

    print(f"  Unnormalized (dataset stats):  {np.round(unnorm, 4)}")

    # Extract position, rotation, gripper
    pos  = unnorm[:3]   # meters
    rot  = unnorm[3:6]  # radians (axis-angle)
    grip = float(unnorm[6])

    # Convert axis-angle rotation to degrees
    coords = [
        round(float(pos[0]) * 1000, 1),          # x: m → mm
        round(float(pos[1]) * 1000, 1),          # y: m → mm
        round(float(pos[2]) * 1000, 1),          # z: m → mm
        round(float(np.degrees(rot[0])), 1),     # rx: rad → deg
        round(float(np.degrees(rot[1])), 1),     # ry: rad → deg
        round(float(np.degrees(rot[2])), 1),     # rz: rad → deg
    ]

    # Gripper: normalize to 0-100%
    g_pct = int(np.clip(
        (grip - float(ACTION_MIN[6])) /
        (float(ACTION_MAX[6]) - float(ACTION_MIN[6]) + 1e-8) * 100,
        0, 100
    ))

    return coords, g_pct


def is_safe(coords):
    """M750 workspace safety check"""
    return (
        -400 <= coords[0] <= 400 and
        -400 <= coords[1] <= 400 and
          50 <= coords[2] <= 600 and
        -180 <= coords[3] <= 180 and
        -180 <= coords[4] <= 180 and
        -180 <= coords[5] <= 180
    )

print("✅ Fixed get_action_v2() defined")
print("   Now uses DATASET stats for unnormalization")

✅ Fixed get_action_v2() defined
   Now uses DATASET stats for unnormalization


In [ ]:
print("=" * 60)
print("🧪 TESTING FIXED PREDICTIONS")
print("=" * 60)

test_frames = [500, 5000, 10000, 20000, 30000, 40000, 50000]
predictions = []
grips       = []

for frame_idx in test_frames:
    print(f"\n--- Frame {frame_idx} ---")
    img_arr   = np.array(root['data']['camera0_rgb'][frame_idx], dtype=np.uint8)
    coords, g = get_action_v2(img_arr)
    safe      = is_safe(coords)

    predictions.append(coords)
    grips.append(g)

    print(f"  pos:  x={coords[0]:>8.1f}mm  "
          f"y={coords[1]:>8.1f}mm  "
          f"z={coords[2]:>8.1f}mm")
    print(f"  rot:  rx={coords[3]:>7.1f}°  "
          f"ry={coords[4]:>7.1f}°  "
          f"rz={coords[5]:>7.1f}°")
    print(f"  grip: {g}%")
    print(f"  {'✅ SAFE' if safe else '❌ UNSAFE'}")

# Diversity analysis
pred_arr = np.array(predictions)
x_range  = pred_arr[:, 0].max() - pred_arr[:, 0].min()
y_range  = pred_arr[:, 1].max() - pred_arr[:, 1].min()
z_range  = pred_arr[:, 2].max() - pred_arr[:, 2].min()
g_range  = max(grips) - min(grips)
n_safe   = sum(is_safe(p) for p in predictions)

print("\n" + "=" * 60)
print("📊 DIVERSITY ANALYSIS")
print("=" * 60)
print(f"  X range:    {x_range:>8.1f}mm  (before: ~6mm,  want: >50mm)")
print(f"  Y range:    {y_range:>8.1f}mm  (before: ~11mm, want: >50mm)")
print(f"  Z range:    {z_range:>8.1f}mm  (before: ~4mm,  want: >20mm)")
print(f"  Grip range: {g_range:>8}%   (before: 0%,   want: >20%)")
print(f"  Safe:       {n_safe}/{len(predictions)}")

print("\n" + "=" * 60)
if x_range > 50 and n_safe >= 4:
    print("✅✅ FULLY FIXED! Predictions are diverse and safe!")
    print("   Run Cell 8 to deploy on robot!")
elif x_range > 20 or n_safe >= 3:
    print("⚠️  Partial fix - some improvement")
    print("   Try running robot with Cell 8")
    print("   But model likely needs retraining")
else:
    print("❌ Still not working")
    print("   Problem is in the model itself")
    print("   MUST RETRAIN with:")
    print("   1. Filtered dataset (only marker task)")
    print("   2. Correct normalizer")
    print("   3. 3K steps with lr=1e-5")
print("=" * 60)

🧪 TESTING FIXED PREDICTIONS

--- Frame 500 ---
  Raw model output (normalized): [-9.9999997e-05  1.2000001e-03  5.0000002e-04  2.8774688e+03
 -1.0100000e+00 -5.9600002e-01 -1.3075841e+02]
  Unnormalized (dataset stats):  [ 0.0077 -0.1061  0.1276  3.1409 -0.9941 -0.0668  0.0065]
  pos:  x=     7.7mm  y=  -106.1mm  z=   127.6mm
  rot:  rx=  180.0°  ry=  -57.0°  rz=   -3.8°
  grip: 0%
  ✅ SAFE

--- Frame 5000 ---
  Raw model output (normalized): [ 3.0000001e-04  0.0000000e+00  0.0000000e+00  2.8052031e+03
 -1.1450000e-01 -4.3090001e-01 -1.3237750e+02]
  Unnormalized (dataset stats):  [ 0.0078 -0.1062  0.1276  3.1409 -0.2565 -0.0359  0.0065]
  pos:  x=     7.8mm  y=  -106.2mm  z=   127.6mm
  rot:  rx=  180.0°  ry=  -14.7°  rz=   -2.1°
  grip: 0%
  ✅ SAFE

--- Frame 10000 ---
  Raw model output (normalized): [ 5.800000e-03  1.630000e-02 -2.800000e-03  4.881638e+02 -2.294700e+00
 -1.809000e+00 -2.574540e+01]
  Unnormalized (dataset stats):  [ 0.0086 -0.104   0.1273  3.1409 -2.0523 -0.2934  0

: 